In [1]:
import fastf1
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ALWAYS the first FastF1 call — before anything else
fastf1.Cache.enable_cache('../cache')

print("FastF1 version:", fastf1.__version__)
print("Cache enabled.")

FastF1 version: 3.8.2
Cache enabled.


In [2]:
# Look at the 2023 season schedule first
schedule_2023 = fastf1.get_event_schedule(2023)
print(schedule_2023[['RoundNumber', 'EventName', 'EventDate', 'EventFormat']].to_string())

    RoundNumber                 EventName  EventDate      EventFormat
0             0        Pre-Season Testing 2023-02-25          testing
1             1        Bahrain Grand Prix 2023-03-05     conventional
2             2  Saudi Arabian Grand Prix 2023-03-19     conventional
3             3     Australian Grand Prix 2023-04-02     conventional
4             4     Azerbaijan Grand Prix 2023-04-30  sprint_shootout
5             5          Miami Grand Prix 2023-05-07     conventional
6             6         Monaco Grand Prix 2023-05-28     conventional
7             7        Spanish Grand Prix 2023-06-04     conventional
8             8       Canadian Grand Prix 2023-06-18     conventional
9             9       Austrian Grand Prix 2023-07-02  sprint_shootout
10           10        British Grand Prix 2023-07-09     conventional
11           11      Hungarian Grand Prix 2023-07-23     conventional
12           12        Belgian Grand Prix 2023-07-30  sprint_shootout
13           13     

In [3]:
# Test fetch — 2023 Bahrain GP
session_race = fastf1.get_session(2023, 1, 'R')
session_race.load(telemetry=False, laps=False, weather=True)
# telemetry=False and laps=False dramatically speeds up loading
# weather=True because you need rain data

print(type(session_race.results))
print(session_race.results.columns.tolist())
print(session_race.results.head())

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']


<class 'fastf1.core.SessionResults'>
['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps']
   DriverNumber BroadcastName Abbreviation        DriverId         TeamName  \
1             1  M VERSTAPPEN          VER  max_verstappen  Red Bull Racing   
11           11       S PEREZ          PER           perez  Red Bull Racing   
14           14      F ALONSO          ALO          alonso     Aston Martin   
55           55       C SAINZ          SAI           sainz          Ferrari   
44           44    L HAMILTON          HAM        hamilton         Mercedes   

   TeamColor        TeamId FirstName    LastName         FullName  ...  \
1     3671C6      red_bull       Max  Verstappen   Max Verstappen  ...   
11    3671C6      red_bull    Sergio       Perez     Sergio Perez  ...  

In [4]:
# See exactly what columns you have
print(session_race.results.dtypes)
print("\nSample row:")
print(session_race.results.iloc[0])

DriverNumber                   object
BroadcastName                  object
Abbreviation                   object
DriverId                       object
TeamName                       object
TeamColor                      object
TeamId                         object
FirstName                      object
LastName                       object
FullName                       object
HeadshotUrl                    object
CountryCode                    object
Position                      float64
ClassifiedPosition             object
GridPosition                  float64
Q1                    timedelta64[ns]
Q2                    timedelta64[ns]
Q3                    timedelta64[ns]
Time                  timedelta64[ns]
Status                         object
Points                        float64
Laps                          float64
dtype: object

Sample row:
DriverNumber                                                          1
BroadcastName                                              M VERS

In [5]:
session_quali = fastf1.get_session(2023, 1, 'Q')
session_quali.load(telemetry=False, laps=True, weather=False)
# laps=True needed for qualifying times

# Get the best lap time per driver
quali_times = session_quali.laps.groupby('Driver')['LapTime'].min().reset_index()
quali_times.columns = ['Abbreviation', 'QualiTime']
quali_times['QualiTime_seconds'] = quali_times['QualiTime'].dt.total_seconds()

# Pole time = minimum lap time
pole_time = quali_times['QualiTime_seconds'].min()
quali_times['QualiGapToPole_pct'] = (
    (quali_times['QualiTime_seconds'] - pole_time) / pole_time * 100
)

print(quali_times.sort_values('QualiTime_seconds'))

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 

   Abbreviation              QualiTime  QualiTime_seconds  QualiGapToPole_pct
18          VER 0 days 00:01:29.708000             89.708            0.000000
11          PER 0 days 00:01:29.846000             89.846            0.153832
7           LEC        0 days 00:01:30             90.000            0.325501
14          SAI 0 days 00:01:30.154000             90.154            0.497169
1           ALO 0 days 00:01:30.336000             90.336            0.700049
13          RUS 0 days 00:01:30.340000             90.340            0.704508
5           HAM 0 days 00:01:30.384000             90.384            0.753556
6           HUL 0 days 00:01:30.809000             90.809            1.227315
16          STR 0 days 00:01:30.836000             90.836            1.257413
10          OCO 0 days 00:01:30.914000             90.914            1.344362
9           NOR 0 days 00:01:31.381000             91.381            1.864940
17          TSU 0 days 00:01:31.400000             91.400       

In [6]:
weather = session_race.weather_data
print(weather.columns.tolist())
print(weather.head())

# Rainfall column is boolean — True means rain detected
# A race is "wet" if it rained at any point
is_wet = int(weather['Rainfall'].any())
print(f"Wet race: {is_wet}")

['Time', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed']
                    Time  AirTemp  Humidity  Pressure  Rainfall  TrackTemp  \
0 0 days 00:00:45.438000     29.8      19.0    1016.5     False       35.1   
1 0 days 00:01:45.453000     29.7      19.0    1016.5     False       35.0   
2 0 days 00:02:45.452000     29.7      19.0    1016.5     False       34.9   
3 0 days 00:03:45.466000     29.6      19.0    1016.5     False       34.9   
4 0 days 00:04:45.449000     29.6      19.0    1016.5     False       34.8   

   WindDirection  WindSpeed  
0            176        1.2  
1            182        1.2  
2            156        1.1  
3            201        0.8  
4            219        0.8  
Wet race: 0


In [7]:
# Race control messages contain safety car info
rcm = session_race.race_control_messages
print(rcm.columns.tolist())
print(rcm[rcm['Message'].str.contains('SAFETY CAR', na=False)])

# Binary: did a safety car deploy at any point?
safety_car = int(rcm['Message'].str.contains('SAFETY CAR', na=False).any())
print(f"Safety car deployed: {safety_car}")

['Time', 'Category', 'Message', 'Status', 'Flag', 'Scope', 'Sector', 'RacingNumber', 'Lap']
                  Time   Category                      Message    Status  \
40 2023-03-05 16:10:19  SafetyCar  VIRTUAL SAFETY CAR DEPLOYED  DEPLOYED   
42 2023-03-05 16:11:55  SafetyCar    VIRTUAL SAFETY CAR ENDING    ENDING   

    Flag Scope  Sector RacingNumber  Lap  
40  None  None     NaN         None   41  
42  None  None     NaN         None   42  
Safety car deployed: 1


In [8]:
race_results = session_race.results[['Abbreviation', 'TeamName', 
                                      'GridPosition', 'Position', 
                                      'Points', 'Status']].copy()

# Add race metadata
race_results['Year'] = 2023
race_results['Round'] = 1
race_results['EventName'] = session_race.event['EventName']
race_results['CircuitName'] = session_race.event['Location']

# Add quali gap
race_results = race_results.merge(
    quali_times[['Abbreviation', 'QualiGapToPole_pct']], 
    on='Abbreviation', 
    how='left'
)

# Add contextual flags
race_results['IsWet'] = is_wet
race_results['SafetyCarDeployed'] = safety_car

# Clean position — DNFs have NaN, fill with 20 (last place proxy)
race_results['Position'] = pd.to_numeric(race_results['Position'], errors='coerce')
race_results['GridPosition'] = pd.to_numeric(race_results['GridPosition'], errors='coerce')

# DNF flag
race_results['IsDNF'] = race_results['Status'].apply(
    lambda x: 0 if str(x) in ['Finished', '+1 Lap', '+2 Laps', '+3 Laps', 
                                '+4 Laps', '+5 Laps'] else 1
)

print(race_results)

   Abbreviation         TeamName  GridPosition  Position  Points    Status  \
0           VER  Red Bull Racing           1.0       1.0    25.0  Finished   
1           PER  Red Bull Racing           2.0       2.0    18.0  Finished   
2           ALO     Aston Martin           5.0       3.0    15.0  Finished   
3           SAI          Ferrari           4.0       4.0    12.0  Finished   
4           HAM         Mercedes           7.0       5.0    10.0  Finished   
5           STR     Aston Martin           8.0       6.0     8.0  Finished   
6           RUS         Mercedes           6.0       7.0     6.0  Finished   
7           BOT       Alfa Romeo          12.0       8.0     4.0  Finished   
8           GAS           Alpine          20.0       9.0     2.0  Finished   
9           ALB         Williams          15.0      10.0     1.0  Finished   
10          TSU       AlphaTauri          14.0      11.0     0.0  Finished   
11          SAR         Williams          16.0      12.0     0.0

In [ ]:
#FULL DATA COLLECTION STARTS 

In [11]:
def fetch_race_data(year, round_num):
    """
    Fetches and merges race results + qualifying times + 
    weather + safety car for a single race.
    Returns a DataFrame of ~20 rows (one per driver).
    Returns None if the session fails to load.
    """
    try:
        # --- Race session ---
        race = fastf1.get_session(year, round_num, 'R')
        race.load(telemetry=False, laps=False, weather=True)
        
        results = race.results[['Abbreviation', 'TeamName',
                                  'GridPosition', 'Position',
                                  'Points', 'Status']].copy()
        
        results['Year'] = year
        results['Round'] = round_num
        results['EventName'] = race.event['EventName']
        results['CircuitName'] = race.event['Location']
        
        # Weather
        results['IsWet'] = int(race.weather_data['Rainfall'].any())
        
        # Safety car
        rcm = race.race_control_messages
        results['SafetyCarDeployed'] = int(
            rcm['Message'].str.contains('SAFETY CAR', na=False).any()
        )
        
        # DNF flag
        finished_statuses = ['Finished', '+1 Lap', '+2 Laps', '+3 Laps',
                              '+4 Laps', '+5 Laps', '+6 Laps']
        results['IsDNF'] = results['Status'].apply(
            lambda x: 0 if str(x) in finished_statuses else 1
        )
        
        # Clean numeric columns
        results['Position'] = pd.to_numeric(results['Position'], errors='coerce')
        results['GridPosition'] = pd.to_numeric(results['GridPosition'], errors='coerce')
        
        # --- Qualifying session ---
        try:
            quali = fastf1.get_session(year, round_num, 'Q')
            quali.load(telemetry=False, laps=True, weather=False)
            
            q_times = (quali.laps
                           .groupby('Driver')['LapTime']
                           .min()
                           .reset_index())
            q_times.columns = ['Abbreviation', 'QualiTime']
            q_times['QualiTime_s'] = q_times['QualiTime'].dt.total_seconds()
            
            pole = q_times['QualiTime_s'].min()
            q_times['QualiGapToPole_pct'] = (
                (q_times['QualiTime_s'] - pole) / pole * 100
            )
            
            results = results.merge(
                q_times[['Abbreviation', 'QualiGapToPole_pct']],
                on='Abbreviation',
                how='left'
            )
        except Exception:
            # Some rounds have no standard qualifying (Sprint weekends)
            results['QualiGapToPole_pct'] = np.nan
        
        return results
    
    except Exception as e:
        print(f"  FAILED: {year} Round {round_num} — {e}")
        return None

In [12]:
all_data = []
years = [2018, 2019, 2020, 2021, 2022, 2023]

for year in years:
    print(f"\n{'='*40}")
    print(f"Fetching {year} season...")
    
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    
    # Filter to race rounds only (Round > 0)
    race_rounds = schedule[schedule['RoundNumber'] > 0]['RoundNumber'].tolist()
    
    for rnd in tqdm(race_rounds, desc=f"{year}"):
        df = fetch_race_data(year, rnd)
        if df is not None:
            all_data.append(df)

# Combine everything
master_df = pd.concat(all_data, ignore_index=True)
print(f"\nTotal rows: {len(master_df)}")
print(f"Total races: {master_df.groupby(['Year','Round']).ngroups}")
print(master_df.head())


Fetching 2018 season...


2018:   0%|                                             | 0/21 [00:00<?, ?it/s]core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['5', '44', '7', '3', '14', '33', '27', '77', '2', '55', '11', '3

  FAILED: 2018 Round 19 — The data you are trying to access has not been loaded yet. See `Session.load`


core           INFO 	Loading data for Brazilian Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['44', '33', '7', '3', '77', '5', '16', '8', '20', '11', '28', '55', '10', '2', '31', '35', '14', '18', '27', '9']
core           INFO 	Loading d


Fetching 2019 season...


...

2019:  95%|██████████████████████████████████▎ | 20/21 [04:17<00:09,  9.75s/it]core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


  FAILED: 2019 Round 20 — any API: 500 calls/h


2019: 100%|████████████████████████████████████| 21/21 [04:17<00:00, 12.28s/it]

  FAILED: 2019 Round 21 — any API: 500 calls/h

Fetching 2020 season...


RateLimitExceededError: any API: 500 calls/h

In [13]:
import pandas as pd

master_df = pd.concat(all_data, ignore_index=True)
print(f"Rows collected so far: {len(master_df)}")
print(master_df.groupby('Year').size())
print(f"Last successful: Year {master_df['Year'].max()}, Round {master_df[master_df['Year']==master_df['Year'].max()]['Round'].max()}")

Rows collected so far: 740
Year
2018    400
2019    340
dtype: int64
Last successful: Year 2019, Round 18


In [14]:
master_df.to_csv('../data/raw/f1_raw_partial.csv', index=False)
print("Saved partial data")

Saved partial data


In [15]:
import time

def fetch_race_data(year, round_num):
    try:
        race = fastf1.get_session(year, round_num, 'R')
        race.load(telemetry=False, laps=False, weather=True)
        
        results = race.results[['Abbreviation', 'TeamName',
                                  'GridPosition', 'Position',
                                  'Points', 'Status']].copy()
        
        results['Year'] = year
        results['Round'] = round_num
        results['EventName'] = race.event['EventName']
        results['CircuitName'] = race.event['Location']
        
        results['IsWet'] = int(race.weather_data['Rainfall'].any())
        
        rcm = race.race_control_messages
        results['SafetyCarDeployed'] = int(
            rcm['Message'].str.contains('SAFETY CAR', na=False).any()
        )
        
        finished_statuses = ['Finished', '+1 Lap', '+2 Laps', '+3 Laps',
                              '+4 Laps', '+5 Laps', '+6 Laps']
        results['IsDNF'] = results['Status'].apply(
            lambda x: 0 if str(x) in finished_statuses else 1
        )
        
        results['Position'] = pd.to_numeric(results['Position'], errors='coerce')
        results['GridPosition'] = pd.to_numeric(results['GridPosition'], errors='coerce')
        
        # Qualifying
        try:
            quali = fastf1.get_session(year, round_num, 'Q')
            quali.load(telemetry=False, laps=True, weather=False)
            
            q_times = (quali.laps
                           .groupby('Driver')['LapTime']
                           .min()
                           .reset_index())
            q_times.columns = ['Abbreviation', 'QualiTime']
            q_times['QualiTime_s'] = q_times['QualiTime'].dt.total_seconds()
            pole = q_times['QualiTime_s'].min()
            q_times['QualiGapToPole_pct'] = (
                (q_times['QualiTime_s'] - pole) / pole * 100
            )
            results = results.merge(
                q_times[['Abbreviation', 'QualiGapToPole_pct']],
                on='Abbreviation', how='left'
            )
        except Exception:
            results['QualiGapToPole_pct'] = np.nan
        
        return results
    
    except Exception as e:
        print(f"  FAILED: {year} Round {round_num} — {e}")
        return None

In [16]:
import time

# Load what you already saved
existing = pd.read_csv('../data/raw/f1_raw_partial.csv')
all_data = [existing]
print(f"Loaded {len(existing)} existing rows covering 2018 + 2019 partial")

# Resume from 2019 Round 19
years_and_starts = {
    2019: 19,
    2020: 1,
    2021: 1,
    2022: 1,
    2023: 1,
}

for year, start_round in years_and_starts.items():
    print(f"\n{'='*40}")
    print(f"Fetching {year} starting from round {start_round}...")
    
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    race_rounds = schedule[
        schedule['RoundNumber'] >= start_round
    ]['RoundNumber'].tolist()
    
    for rnd in tqdm(race_rounds, desc=f"{year}"):
        df = fetch_race_data(year, rnd)
        if df is not None:
            all_data.append(df)
        time.sleep(3)
    
    # Save checkpoint after every full season
    checkpoint = pd.concat(all_data, ignore_index=True)
    checkpoint.to_csv('../data/raw/f1_raw_partial.csv', index=False)
    print(f"Checkpoint saved — {len(checkpoint)} rows total")

print("\nAll done!")

Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2019.json failed; using cached response
Traceback (most recent call last):
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests_cache\session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests_cache\session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastf1\req.py", line 110, in send
    lim.limit()
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastf1\req.py", line 80, in limit
    raise RateLimitExceededError

Loaded 740 existing rows covering 2018 + 2019 partial

Fetching 2019 starting from round 19...


2019:   0%|                                              | 0/3 [00:00<?, ?it/s]Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2019.json failed; using cached response
Traceback (most recent call last):
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests_cache\session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests_cache\session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastf1\req.py", line 110, in send
    lim.limit()
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\s

  FAILED: 2019 Round 19 — any API: 500 calls/h


2019:  33%|████████████▋                         | 1/3 [00:03<00:06,  3.49s/it]Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2019.json failed; using cached response
Traceback (most recent call last):
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests_cache\session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests_cache\session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastf1\req.py", line 110, in send
    lim.limit()
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\s

...

2019:  67%|█████████████████████████▎            | 2/3 [00:06<00:03,  3.35s/it]Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2019.json failed; using cached response
Traceback (most recent call last):
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests_cache\session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests_cache\session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastf1\req.py", line 110, in send
    lim.limit()
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\s

  FAILED: 2019 Round 21 — any API: 500 calls/h


2019: 100%|██████████████████████████████████████| 3/3 [00:09<00:00,  3.33s/it]

Checkpoint saved — 740 rows total

Fetching 2020 starting from round 1...


RateLimitExceededError: any API: 500 calls/h

In [17]:
# Hardcoded round counts — no API call needed
# Verified against official F1 records
SEASON_ROUNDS = {
    2018: 21,
    2019: 21,
    2020: 17,
    2021: 22,
    2022: 22,
    2023: 22,
}

print("Season rounds hardcoded — no API calls used")
print(SEASON_ROUNDS)

Season rounds hardcoded — no API calls used
{2018: 21, 2019: 21, 2020: 17, 2021: 22, 2022: 22, 2023: 22}


In [18]:
# This version NEVER calls get_event_schedule
# Saves ~6 API calls (one per season)

years_and_starts = {
    2019: 19,
    2020: 1,
    2021: 1,
    2022: 1,
    2023: 1,
}

for year, start_round in years_and_starts.items():
    print(f"\n{'='*40}")
    print(f"Fetching {year} starting from round {start_round}...")
    
    total_rounds = SEASON_ROUNDS[year]
    race_rounds = list(range(start_round, total_rounds + 1))
    
    for rnd in tqdm(race_rounds, desc=f"{year}"):
        df = fetch_race_data(year, rnd)
        if df is not None:
            all_data.append(df)
        time.sleep(5)  # increased to 5s to be safer
    
    checkpoint = pd.concat(all_data, ignore_index=True)
    checkpoint.to_csv('../data/raw/f1_raw_partial.csv', index=False)
    print(f"Checkpoint saved — {len(checkpoint)} rows total")

print("\nAll done!")


Fetching 2019 starting from round 19...


2019:   0%|                                              | 0/3 [00:00<?, ?it/s]core           INFO 	Loading data for United States Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['77', '44', '33', '16', '23', '3', '4', '55', '

Checkpoint saved — 800 rows total

Fetching 2020 starting from round 1...


2020:   0%|                                             | 0/17 [00:00<?, ?it/s]core           INFO 	Loading data for Austrian Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['77', '16', '4', '44', '55', '11', '10', '31', '99', '5', '6', '26

  FAILED: 2020 Round 16 — The data you are trying to access has not been loaded yet. See `Session.load`


...

  FAILED: 2023 Round 21 — any API: 500 calls/h


2023:  95%|██████████████████████████████████▎ | 21/22 [01:51<00:05,  5.26s/it]Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2023.json failed; using cached response
Traceback (most recent call last):
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests_cache\session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests_cache\session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastf1\req.py", line 110, in send
    lim.limit()
  File "C:\Users\akshat bhaskar\AppData\Local\Programs\Python\Python312\Lib\s

  FAILED: 2023 Round 22 — any API: 500 calls/h


2023: 100%|████████████████████████████████████| 22/22 [01:56<00:00,  5.32s/it]

Checkpoint saved — 1500 rows total

All done!


In [19]:
# What you have is your final dataset — rename it
import shutil
shutil.copy(
    '../data/raw/f1_raw_partial.csv',
    '../data/raw/f1_raw_2018_2021.csv'
)
print("Final dataset locked as f1_raw_2018_2021.csv")

Final dataset locked as f1_raw_2018_2021.csv


In [20]:
# Check 1 — DNF rate (expect ~15-20%)
print(f"DNF rate: {df['IsDNF'].mean():.2%}")

# Check 2 — Grid position range
print(f"\nGrid position range: {df['GridPosition'].min()} to {df['GridPosition'].max()}")

# Check 3 — Wet race count
wet = df.groupby(['Year','Round'])['IsWet'].first()
print(f"\nWet races: {wet.sum()} out of {len(wet)} total")

# Check 4 — QualiGap coverage
print(f"\nQualiGap available: {df['QualiGapToPole_pct'].notna().mean():.2%}")

# Check 5 — Sample rows
print(f"\nSample:")
print(df.head(3).to_string())

TypeError: 'NoneType' object is not subscriptable

In [21]:
import pandas as pd

# Load the newly locked dataset into the 'df' variable
file_path = '../data/raw/f1_raw_2018_2021.csv'
df = pd.read_csv(file_path)

# Verify it loaded correctly
print(f"Successfully loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")

Successfully loaded dataset with 1500 rows and 14 columns.


In [22]:
# Check 1 — DNF rate (expect ~15-20%)
print(f"DNF rate: {df['IsDNF'].mean():.2%}")

# Check 2 — Grid position range
print(f"\nGrid position range: {df['GridPosition'].min()} to {df['GridPosition'].max()}")

# Check 3 — Wet race count
wet = df.groupby(['Year','Round'])['IsWet'].first()
print(f"\nWet races: {wet.sum()} out of {len(wet)} total")

# Check 4 — QualiGap coverage
print(f"\nQualiGap available: {df['QualiGapToPole_pct'].notna().mean():.2%}")

# Check 5 — Sample rows
print(f"\nSample:")
print(df.head(3).to_string())

DNF rate: 16.47%

Grid position range: 0.0 to 20.0

Wet races: 22 out of 75 total

QualiGap available: 85.67%

Sample:
  Abbreviation  TeamName  GridPosition  Position  Points    Status  Year  Round              EventName CircuitName  IsWet  SafetyCarDeployed  IsDNF  QualiGapToPole_pct
0          VET   Ferrari           3.0       1.0    25.0  Finished  2018      1  Australian Grand Prix   Melbourne      1                  1      0            0.830417
1          HAM  Mercedes           1.0       2.0    18.0  Finished  2018      1  Australian Grand Prix   Melbourne      1                  1      0            0.000000
2          RAI   Ferrari           2.0       3.0    15.0  Finished  2018      1  Australian Grand Prix   Melbourne      1                  1      0            0.818097
